In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import timedelta

In [ ]:
path = "./"
df = pd.read_csv(os.path.join(path,"ventanas.csv"))

In [ ]:
df

In [ ]:
df.groupby('key')['label'].sum().sort_values(ascending=False)


# XGBoost


In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Dividir por paciente
pacientes = df['key'].unique()
train_keys, test_keys = train_test_split(pacientes, test_size=0.2, random_state=42)

df_train = df[df['key'].isin(train_keys)]
df_test = df[df['key'].isin(test_keys)]

X_train = df_train.drop(columns=['key', 'start_date', 'end_date', 'label'])
y_train = df_train['label']
X_test = df_test.drop(columns=['key', 'start_date', 'end_date', 'label'])
y_test = df_test['label']

# Calcular el peso de la clase minoritaria
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_pos_weight = num_neg / num_pos if num_pos > 0 else 1

# Entrenar modelo XGBoost con ajuste de desbalance
model_balanced = XGBClassifier(eval_metric='logloss',
                               random_state=42,
                               scale_pos_weight=scale_pos_weight)

X_train_clean = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test_clean = X_test.fillna(0).replace([np.inf, -np.inf], 0)

model_balanced.fit(X_train_clean, y_train)

# Predicción
y_pred_balanced = model_balanced.predict(X_test_clean)

# Resultados
report_bal = classification_report(y_test, y_pred_balanced, output_dict=True)
conf_matrix_bal = confusion_matrix(y_test, y_pred_balanced)

# Mostrar matriz de confusión
plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix_bal, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - XGBoost (balanced)")
plt.tight_layout()
plt.show()

# Mostrar resumen de métricas
pd.DataFrame(report_bal).transpose()